In [21]:
import os
import numpy as np
import random
import math
import pydicom
from pydicom.dataset import Dataset, FileDataset
from datetime import datetime

In [ ]:
OUTPUT_DIR = "../data/ct_ellipses"
NUM_IMAGES = 100
IMG_SIZE = (128, 128)  # (rows, cols)

NUM_ELLIPSES_RANGE = (1, 5)
AXIS_RANGE = (10, 60)

HU_BACKGROUND = -1000  # air
HU_RANGE = [
    (30, 80),      # tkanka miękka
    (300, 800)     # kość
]

PIXEL_SPACING = [1.0, 1.0]

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)


def add_rotated_ellipse(image, center, axes, angle, value):
    cy, cx = center
    a, b = axes
    theta = math.radians(angle)

    rows, cols = image.shape
    y, x = np.ogrid[:rows, :cols]

    x0 = x - cx
    y0 = y - cy

    xr = x0 * math.cos(theta) + y0 * math.sin(theta)
    yr = -x0 * math.sin(theta) + y0 * math.cos(theta)

    mask = (xr**2) / (a**2) + (yr**2) / (b**2) <= 1
    image[mask] = value

In [ ]:
for i in range(NUM_IMAGES):
    img = np.full(IMG_SIZE, HU_BACKGROUND, dtype=np.int16)

    for _ in range(random.randint(*NUM_ELLIPSES_RANGE)):
        center = (
            random.randint(0, IMG_SIZE[0] - 1),
            random.randint(0, IMG_SIZE[1] - 1),
        )
        axes = (random.randint(*AXIS_RANGE), random.randint(*AXIS_RANGE))
        angle = random.uniform(0, 180)

        hu_min, hu_max = random.choice(HU_RANGE)
        value = random.randint(hu_min, hu_max)

        add_rotated_ellipse(img, center, axes, angle, value)

    # DICOM DATASET
    file_meta = Dataset()
    file_meta.MediaStorageSOPClassUID = pydicom.uid.CTImageStorage
    file_meta.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
    file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

    ds = FileDataset(
        filename_or_obj=None,
        dataset=Dataset(),
        file_meta=file_meta,
        preamble=b"\0" * 128,
    )

    # --- Identifiers ---
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.StudyInstanceUID = pydicom.uid.generate_uid()
    ds.SeriesInstanceUID = pydicom.uid.generate_uid()

    ds.Modality = "CT"
    ds.ImageType = ["ORIGINAL", "PRIMARY", "AXIAL"]
    ds.PatientName = "PHANTOM^ELLIPSE"
    ds.PatientID = "000001"

    ds.StudyDate = datetime.now().strftime("%Y%m%d")
    ds.StudyTime = datetime.now().strftime("%H%M%S")

    # --- Geometry ---
    ds.Rows, ds.Columns = IMG_SIZE
    ds.PixelSpacing = PIXEL_SPACING
    ds.SliceThickness = 1.0
    ds.ImagePositionPatient = [0.0, 0.0, float(i)]
    ds.ImageOrientationPatient = [1, 0, 0, 0, 1, 0]
    ds.ReconstructionDiameter = max(IMG_SIZE) * PIXEL_SPACING[0]

    # --- Pixel data ---
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 1  # signed int
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"

    # HU scaling
    ds.RescaleIntercept = 0
    ds.RescaleSlope = 1

    ds.PixelData = img.tobytes()

    filename = os.path.join(OUTPUT_DIR, f"CT_{i:06d}.dcm")
    ds.save_as(filename)

print("DICOM CT dataset wygenerowany:", OUTPUT_DIR)

DICOM CT dataset wygenerowany: ../data/ct_ellipses
